In [1]:
# Setting system path and project root
import os
import sys

PROJECT_ROOT_DIR = os.path.abspath('../../')
sys.path.append(PROJECT_ROOT_DIR) # bringing system path to project root

def get_fp(relative_path):
    return os.path.join(PROJECT_ROOT_DIR, relative_path)

In [ ]:
# imports
import os
import glob
from tqdm import tqdm
from collections import Counter
from src.const.llm import ModelAPIConfig
from src.kgqa_tool.llm_request import check_early_stop

In [3]:
llm_config = ModelAPIConfig("gpt-oss-120b", 'http://lola.cs.uni-paderborn.de:9292/v1', '') # LLM to use

In [4]:
# experiment analysis dictionary
analysis_path_dict = {
    'qald9plus_test': ['data_dir/processed_kgqa_ds/qald9plus/test/ablation.test.prediction/tentrisq10_aug_gold/analysis/en__otus__PBSG_MHOP__t20-h1-pc-ausm-t5aug_erl-exlim10-clsinf__gptoss120b'],
    'qald10_test': ['data_dir/processed_kgqa_ds/qald10/test/ablation.test.prediction/tentrisq10_aug_gold/analysis/en__otus__PBSG_MHOP__t20-h5-pc-ausm-t5aug_erl-exlim10__gptoss120b'],
    'lcquad2_test': ['data_dir/processed_kgqa_ds/lcquad2/test/test.prediction/tentrisq10_aug_gold/analysis/en__lola__PBSG_MHOP__t20-h-1-pc-ausm-t5aug_erl-exlim10-clsinf__gptoss120b']
}

In [5]:
def is_early_stop(content):
    llm_resp, think_content = check_early_stop(content, llm_config)
    #print(f'{think_content}\n{llm_resp}\n\n')
    return llm_resp.lower() == 'y'

In [6]:
# For each dataset
    # For each analysis directory
        # For each *_analysis.txt in the directory
            # Fetch the content
            # Ask LLM if the analysis points to early termination (assume predefined boolean function: check_early_stop)
            # if yes, increment the early_stop counter
# print dataset specific percentage of early stopping and an overall macro and micro average as well

In [7]:
def analyse_dataset(dataset_name: str, analysis_dirs: list) -> dict:
    """
    Walks through every ``*_analysis.txt`` file in the supplied directories,
    runs ``check_early_stop`` on the file content and returns a dict with:

        {
            "total":   <number of analysis files examined>,
            "early":   <number flagged as early termination>,
            "percent": <early / total * 100 (float)>,
        }
    """
    total, early = 0, 0

    for rel_dir in analysis_dirs:
        abs_dir = get_fp(rel_dir)               # absolute path to the dir
        # Grab every file that ends with *_analysis.txt (recursively)
        pattern = os.path.join(abs_dir, "**", "*_analysis.txt")
        for file_path in tqdm(glob.glob(pattern, recursive=True),
                              desc=f"Scanning {dataset_name}",
                              unit="file"):
            total += 1
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
            try:
                if is_early_stop(content):
                    early += 1
            except Exception as e:
                print(f"is_early_stop failed for {file_path}: {e}")

    percent = (early / total * 100) if total else 0.0
    return {"total": total, "early": early, "percent": percent}

In [8]:

# Main aggregation over all datasets
dataset_stats = {}
overall_total, overall_early = 0, 0
macro_percent_sum = 0.0   # for macro‑average (simple mean of per‑dataset percents)

for ds_name, dirs in analysis_path_dict.items():
    stats = analyse_dataset(ds_name, dirs)
    dataset_stats[ds_name] = stats

    overall_total += stats["total"]
    overall_early += stats["early"]
    macro_percent_sum += stats["percent"]

# Micro‑average (global early‑stop rate)
micro_percent = (overall_early / overall_total * 100) if overall_total else 0.0

# Macro‑average (average of per‑dataset percentages)
macro_percent = (macro_percent_sum / len(dataset_stats)) if dataset_stats else 0.0

# Pretty‑print the results
print("\n=== Early‑Termination Analysis ===\n")
for ds, stats in dataset_stats.items():
    print(f"\n{ds:>12}: {stats['early']}/{stats['total']} "
          f"early‑stops : {stats['percent']:.2f}%")

print("\nOverall (micro) early‑stop rate: "
      f"{overall_early}/{overall_total} : {micro_percent:.2f}%")
print("Overall (macro) early‑stop rate: "
      f"{macro_percent:.2f}%\n")

Scanning lcquad2_test: 100%|██████████| 2515/2515 [1:00:59<00:00,  1.45s/file]


=== Early‑Termination Analysis ===


qald9plus_test: 29/64 early‑stops : 45.31%

 qald10_test: 115/174 early‑stops : 66.09%

lcquad2_test: 1257/2515 early‑stops : 49.98%

Overall (micro) early‑stop rate: 1401/2753 : 50.89%
Overall (macro) early‑stop rate: 53.79%

